Kontrollera hur vilket BMU MAX hit och medelvärde alla. 

In [ ]:
from collections import Counter

counts = Counter(map(tuple, winners))

print("Max hits:", max(counts.values()))
print("Mean hits:", np.mean(list(counts.values())))

In [ ]:
counts = Counter(map(tuple, winners))

# hitta neuron med flest hits
top_neuron = counts.most_common(1)[0]
print("Top neuron:", top_neuron)
mask = [tuple(w) == top_neuron[0] for w in winners]
df_som[mask].describe()

In [ ]:
from scipy.ndimage import label

threshold = np.percentile(u_matrix, 40)
regions = u_matrix < threshold

labeled_regions, num = label(regions)

plt.imshow(labeled_regions, cmap="tab10")
plt.title(f"Regions found: {num}")
plt.show()

In [ ]:
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from scipy import ndimage as ndi

distance = -u_matrix

coords = peak_local_max(distance, footprint=np.ones((3,3)))
mask = np.zeros(distance.shape, dtype=bool)
mask[tuple(coords.T)] = True

markers, _ = ndi.label(mask)
labels_ws = watershed(distance, markers)

plt.imshow(labels_ws, cmap="tab10")
plt.title("Watershed regions")
plt.show()

In [ ]:
scaled_data = som_clas_200.scaler.transform(df_som)
hits = som_clas_200.som.activation_response(scaled_data)

active = hits > np.percentile(hits, 50)

labeled, n = label(active)

plt.imshow(labeled, cmap="tab10")
plt.title("Data-driven regions")
plt.show()

In [ ]:
# =========================
# CLEAN REGIONS + STRESS LABELING
# =========================

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

FEATURES = ["HR", "HRV_RMSSD", "SC_PH", "SC_RR"]

# -------------------------
# 1. REMOVE SMALL REGIONS (noise)
# -------------------------
min_samples = 100

region_counts = df_som["region"].value_counts()
valid_regions = region_counts[region_counts >= min_samples].index

df_som["region_clean"] = df_som["region"].where(
    df_som["region"].isin(valid_regions),
    -1  # noise
)

print("\n=== REGION COUNTS (CLEANED) ===")
print(df_som["region_clean"].value_counts())


# -------------------------
# 2. COMPUTE REGION CENTROIDS
# -------------------------
region_stats = df_som[df_som["region_clean"] != -1] \
    .groupby("region_clean")[FEATURES].mean()

print("\n=== REGION CENTROIDS ===")
print(region_stats)


# -------------------------
# 3. CLUSTER REGIONS → 3 STATES
# -------------------------
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
region_labels_k = kmeans.fit_predict(region_stats)

region_stats["cluster"] = region_labels_k


# -------------------------
# 4. ORDER CLUSTERS BY STRESS SCORE
# -------------------------
# Stress physiology:
# ↑ SC_RR, ↑ HR, ↑ SC_PH, ↓ HRV_RMSSD

region_stats["stress_score"] = (
    region_stats["SC_RR"] * 3 +
    region_stats["HR"] * 0.015 +
    region_stats["SC_PH"] * 0.3 -
    region_stats["HRV_RMSSD"] * 0.06
)

cluster_order = (
    region_stats.groupby("cluster")["stress_score"]
    .mean()
    .sort_values()
    .index
)

stress_map = {
    cluster_order[0]: "LOW",
    cluster_order[1]: "MEDIUM",
    cluster_order[2]: "HIGH"
}

region_stats["stress"] = region_stats["cluster"].map(stress_map)

print("\n=== REGION → STRESS ===")
print(region_stats[["cluster", "stress_score", "stress"]])


# -------------------------
# 5. MAP BACK TO DATA
# -------------------------
region_to_stress = region_stats["stress"].to_dict()

df_som["stress_label"] = df_som["region_clean"].map(region_to_stress)
df_som["stress_label"] = df_som["stress_label"].fillna("NOISE")


# -------------------------
# 6. FINAL DISTRIBUTION
# -------------------------
print("\n=== FINAL STRESS DISTRIBUTION ===")
print(df_som["stress_label"].value_counts())


# -------------------------
# 7. FINAL VALIDATION
# -------------------------
print("\n=== FINAL FEATURE MEANS PER STRESS ===")
print(df_som.groupby("stress_label")[FEATURES].mean())

In [ ]:
# =========================================
# CELL 1: MAP DATA → BMU → REGION
# =========================================

import numpy as np
import pandas as pd

# --- VIKTIGT ---
# df_som = ORIGINAL DATA (OSKALAD) → används för analys/tolkning
# scaler används endast för att matcha SOM space

# 1. Skala data (EXAKT samma scaling som träning)
scaled_data = som_clas_200.scaler.transform(df_som)

# 2. Hitta BMU för varje datapunkt
# winner(x) returnerar koordinat (i, j) i SOM-griden
winners = np.array([som_clas_200.som.winner(x) for x in scaled_data])

# 3. Välj vilken region-map du vill använda
# 👉 REKOMMENDERAT: watershed
region_map = labels_ws   # <-- från din watershed-cell

# 4. Mappa varje datapunkt till en region
region_labels = np.array([
    region_map[i, j] for i, j in winners
])

# 5. Skapa analys-DataFrame (OSKALAD DATA + region)
df_region = df_som.copy()
df_region["region"] = region_labels

print("Shape:", df_region.shape)
print("Regions:", np.unique(region_labels))

In [ ]:
# =========================================
# FILTER SMALL REGIONS + RE-ANALYSIS
# =========================================

# 1. Räkna regionstorlek
counts = df_region["region"].value_counts()

print("=== ORIGINAL REGION SIZES ===")
print(counts)

# 2. Filtrera bort små regioner (threshold)
threshold = 100   # du kan justera (50–200)

valid_regions = counts[counts > threshold].index

df_clean = df_region[df_region["region"].isin(valid_regions)].copy()

print("\n=== KEPT REGIONS ===")
print(valid_regions)

print("\nNew shape:", df_clean.shape)

# 3. Remappa regioner → 0,1,2...
region_map_clean = {r:i for i, r in enumerate(valid_regions)}
df_clean["region_clean"] = df_clean["region"].map(region_map_clean)

# 4. Ny statistik (DETTA är den viktiga analysen)
print("\n=== CLEAN REGION MEAN ===")
display(df_clean.groupby("region_clean").mean())

print("\n=== CLEAN REGION STD ===")
display(df_clean.groupby("region_clean").std())

print("\n=== CLEAN REGION SIZE ===")
print(df_clean["region_clean"].value_counts())

In [ ]:
# =========================================
# CELL 3: VISUALIZATION
# =========================================

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

feature_names = ["HR", "HRV_RMSSD", "SC_PH", "SC_RR"]

# -------------------------
# 1. PCA PLOT (FEATURE SPACE)
# -------------------------
# 👉 använder OSKALAD data → men standardiserad för PCA

X_scaled = StandardScaler().fit_transform(df_region[feature_names])
X_pca = PCA(n_components=2).fit_transform(X_scaled)

plt.figure(figsize=(8,6))
sns.scatterplot(
    x=X_pca[:,0],
    y=X_pca[:,1],
    hue=df_region["region"],   # färg = region
    palette="tab10",
    s=10
)

plt.title("PCA of data colored by SOM regions")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.show()


# -------------------------
# 2. BMU MAP (SOM SPACE)
# -------------------------
# 👉 visar VAR datapunkter hamnar i kartan

plt.figure(figsize=(6,6))

plt.scatter(
    winners[:,1],   # x = kolumn
    winners[:,0],   # y = rad
    c=region_labels,
    cmap="tab10",
    s=5
)

plt.gca().invert_yaxis()
plt.title("BMU positions colored by region (SOM space)")
plt.xlabel("SOM X")
plt.ylabel("SOM Y")
plt.show()


# -------------------------
# 3. REGION + HITS (BONUS)
# -------------------------
# 👉 kombinerar regioner + hur mycket data

from collections import Counter

counts = Counter(map(tuple, winners))

hits = np.zeros((som_clas_200.x, som_clas_200.y))

for (i, j), c in counts.items():
    hits[i, j] = c

plt.figure(figsize=(6,6))

plt.imshow(region_map, cmap="tab10", alpha=0.6)
plt.imshow(hits, cmap="hot", alpha=0.4)

plt.title("Regions + Data Density")
plt.colorbar(label="Hit intensity")
plt.show()

In [ ]:
# =========================================
# FINAL VISUALIZATION (CLEAN REGIONS)
# =========================================

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from collections import Counter

feature_names = ["HR", "HRV_RMSSD", "SC_PH", "SC_RR"]

# --------------------------------------------------
# 1. PCA PLOT (FEATURE SPACE)
# --------------------------------------------------
# 👉 INPUT: df_clean (OSKALAD data)
# 👉 Scaling görs endast för PCA

X_scaled = StandardScaler().fit_transform(df_clean[feature_names])
X_pca = PCA(n_components=2).fit_transform(X_scaled)

plt.figure(figsize=(8,6))
sns.scatterplot(
    x=X_pca[:,0],
    y=X_pca[:,1],
    hue=df_clean["region_clean"],   # 🔥 NY REGION
    palette="Set2",
    s=10
)

plt.title("PCA (Clean SOM Regions)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Region")
plt.show()


# --------------------------------------------------
# 2. BMU MAP (SOM SPACE)
# --------------------------------------------------
# 👉 INPUT: winners från tidigare (ALLA datapunkter)
# 👉 filtrerar bara de datapunkter som är kvar i df_clean

mask = df_region["region"].isin(df_clean["region"])

winners_clean = winners[mask]
regions_clean = df_clean["region_clean"].values

plt.figure(figsize=(6,6))

plt.scatter(
    winners_clean[:,1],
    winners_clean[:,0],
    c=regions_clean,
    cmap="bwr",
    s=5
)

plt.gca().invert_yaxis()
plt.title("BMU positions (Clean Regions)")
plt.xlabel("SOM X")
plt.ylabel("SOM Y")
plt.show()


# --------------------------------------------------
# 3. HIT MAP (DATA DENSITY)
# --------------------------------------------------
# 👉 visar hur mycket data varje neuron har

counts = Counter(map(tuple, winners_clean))

hits_clean = np.zeros((som_clas_200.x, som_clas_200.y))

for (i, j), c in counts.items():
    hits_clean[i, j] = c

plt.figure(figsize=(6,6))

plt.imshow(hits_clean, cmap="hot", alpha=0.8)
plt.colorbar(label="Hits")
plt.title("Hit Map (Clean Data Only)")
plt.show()


# --------------------------------------------------
# 4. REGION OVERLAY (STRUCTURE + DATA)
# --------------------------------------------------
# 👉 visar regions + data density tillsammans

plt.figure(figsize=(6,6))

# region_map är ORIGINAL watershed (men vi använder färg efter clean)
region_overlay = np.full_like(region_map, np.nan)

for r_old, r_new in zip(df_clean["region"], df_clean["region_clean"]):
    region_overlay[region_map == r_old] = r_new

plt.imshow(region_overlay, cmap="Set2", alpha=0.6)
plt.imshow(hits_clean, cmap="hot", alpha=0.4)

plt.title("Clean Regions + Data Density")
plt.colorbar(label="Hit intensity")
plt.show()

In [ ]:
# =========================================
# B: CONFIDENCE / UNCERTAINTY
# =========================================

# sannolikhet från modellen
probs = clf.predict_proba(X_pca)

# confidence = max probability
confidence = np.max(probs, axis=1)

# lägg till i dataframe
df_clean["confidence"] = confidence

print("=== CONFIDENCE STATS ===")
print(df_clean["confidence"].describe())

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    X_pca[:,0],
    X_pca[:,1],
    c=confidence,
    cmap="viridis",
    s=10
)

plt.colorbar(label="Confidence")
plt.title("Model Confidence (PCA space)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()